# ⚽ Soccer Player Pizza Chart Comparator

Load the Big-5 leagues stats CSV, search for **Player A**, draw their pizza chart,
then click **"Add Player B"** to overlay a second player's chart on top for comparison.

Built with [mplsoccer](https://mplsoccer.readthedocs.io/) `PyPizza` + `ipywidgets`.


In [ ]:
# 1. Install dependencies (run once)
# !pip install mplsoccer ipywidgets pandas numpy --quiet


In [1]:
# 2. Imports
import pandas as pd
import numpy as np
from mplsoccer import PyPizza
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


In [2]:
# 3. Load and clean the data
CSV_PATH = "big5_player_stats.csv"

# FBref exports have a 2-row header -> real column names are on row index 1
df = pd.read_csv(CSV_PATH, header=1)

# Drop the repeated "Matches" helper column/rows if present
df = df[df["Player"] != "Player"].copy()

# Convert numeric columns
numeric_cols = ["Age", "Born", "MP", "Starts", "Min", "90s", "Gls", "Ast", "G+A",
                "G-PK", "PK", "PKatt", "CrdY", "CrdR",
                "Gls.1", "Ast.1", "G+A.1", "G-PK.1", "G+A-PK"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["Player", "90s"])

# Primary position (first one listed, e.g. "MF,FW" -> "MF")
df["PrimaryPos"] = df["Pos"].str.split(",").str[0]

print(f"Loaded {len(df)} player rows.")
df.head(3)


Loaded 2839 player rows.


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,PrimaryPos
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,0,3,0,0.15,0.18,0.33,0.15,0.33,Matches,MF
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,MF
2,3,Zach Abbott,eng ENG,DF,Nottingham,eng Premier League,19.0,2006.0,3,2,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,DF


In [3]:
# 4. Define the stat set used for the pizza chart and a friendly rename map
# (You can edit this list to use any numeric columns available in the CSV)

STATS = ["Gls.1", "Ast.1", "G+A.1", "G-PK.1", "Min", "Starts"]

STAT_LABELS = {
    "Gls.1":  "Goals\n/90",
    "Ast.1":  "Assists\n/90",
    "G+A.1":  "G+A\n/90",
    "G-PK.1": "Non-Pen\nGoals/90",
    "Min":    "Minutes\nPlayed",
    "Starts": "Starts",
}

MIN_90S = 5  # ignore players with very few minutes when building percentile pool


In [4]:
# 5. Helper functions

def find_player(name):
    """Case-insensitive partial match lookup. Returns the matching row (Series) or None."""
    matches = df[df["Player"].str.contains(name, case=False, na=False)]
    if matches.empty:
        return None
    if len(matches) > 1:
        # Prefer exact (case-insensitive) match if one exists
        exact = matches[matches["Player"].str.lower() == name.lower()]
        if not exact.empty:
            return exact.iloc[0]
        print(f"Multiple matches for '{name}': {matches['Player'].tolist()[:8]}"
              f"{' ...' if len(matches) > 8 else ''}. Using the first match: {matches.iloc[0]['Player']}")
    return matches.iloc[0]


def get_percentiles(player_row):
    """Percentile (0-100) of this player on each STAT, relative to same primary position
    among players with >= MIN_90S 90s played."""
    pos = player_row["PrimaryPos"]
    pool = df[(df["PrimaryPos"] == pos) & (df["90s"] >= MIN_90S)]
    if len(pool) < 10:  # fallback to all positions if pool too small
        pool = df[df["90s"] >= MIN_90S]

    percentiles = []
    for stat in STATS:
        value = player_row[stat]
        pct = (pool[stat] < value).mean() * 100
        percentiles.append(round(pct, 1))
    return percentiles


def get_raw_values(player_row):
    return [round(player_row[s], 2) for s in STATS]


In [5]:
# 6. Plotting functions

def plot_single_pizza(player_row):
    params = [STAT_LABELS[s] for s in STATS]
    values = get_percentiles(player_row)
    raw = get_raw_values(player_row)

    baker = PyPizza(
        params=params,
        background_color="#1a1a1a",
        straight_line_color="#FFFFFF",
        straight_line_lw=1,
        last_circle_lw=1,
        last_circle_color="#FFFFFF",
        other_circle_ls="-.",
        other_circle_lw=1,
    )

    fig, ax = baker.make_pizza(
        values,
        figsize=(8, 8.5),
        param_location=110,
        color_blank_space="same",
        slice_colors=["#1A78CF"] * len(params),
        value_colors=["#FFFFFF"] * len(params),
        value_bck_colors=["#1A78CF"] * len(params),
        kwargs_slices=dict(edgecolor="#FFFFFF", zorder=2, linewidth=1),
        kwargs_params=dict(color="#FFFFFF", fontsize=11, va="center"),
        kwargs_values=dict(
            color="#FFFFFF", fontsize=11, zorder=3,
            bbox=dict(edgecolor="#FFFFFF", facecolor="#1A78CF",
                      boxstyle="round,pad=0.2", lw=1)
        ),
    )

    title = f"{player_row['Player']}  |  {player_row['Squad']}  ({player_row['PrimaryPos']})"
    fig.text(0.5, 0.97, title, ha="center", color="#FFFFFF", fontsize=15, fontweight="bold")
    fig.text(0.5, 0.94,
              "Percentile rank vs. players in the same position  |  Big-5 European Leagues",
              ha="center", color="#CCCCCC", fontsize=9)
    plt.show()


def plot_comparison_pizza(player_a_row, player_b_row):
    params = [STAT_LABELS[s] for s in STATS]
    values_a = get_percentiles(player_a_row)
    values_b = get_percentiles(player_b_row)

    baker = PyPizza(
        params=params,
        background_color="#1a1a1a",
        straight_line_color="#FFFFFF",
        straight_line_lw=1,
        last_circle_lw=1,
        last_circle_color="#FFFFFF",
        other_circle_ls="-.",
        other_circle_lw=1,
    )

    fig, ax = baker.make_pizza(
        values_a,
        compare_values=values_b,
        figsize=(8.5, 9),
        param_location=110,
        color_blank_space="same",
        slice_colors=["#1A78CF"] * len(params),
        kwargs_slices=dict(edgecolor="#FFFFFF", zorder=2, linewidth=1, alpha=.6),
        kwargs_compare=dict(facecolor="#EE7300", edgecolor="#FFFFFF", zorder=2, linewidth=1, alpha=.6),
        kwargs_params=dict(color="#FFFFFF", fontsize=11, va="center"),
        kwargs_values=dict(
            color="#FFFFFF", fontsize=10, zorder=3,
            bbox=dict(edgecolor="#FFFFFF", facecolor="#1A78CF",
                      boxstyle="round,pad=0.2", lw=1)
        ),
        kwargs_compare_values=dict(
            color="#FFFFFF", fontsize=10, zorder=3,
            bbox=dict(edgecolor="#FFFFFF", facecolor="#EE7300",
                      boxstyle="round,pad=0.2", lw=1)
        ),
    )

    title = f"{player_a_row['Player']}  vs  {player_b_row['Player']}"
    fig.text(0.5, 0.98, title, ha="center", color="#FFFFFF", fontsize=16, fontweight="bold")
    fig.text(0.5, 0.95,
              "Percentile rank vs. players in the same position  |  Big-5 European Leagues",
              ha="center", color="#CCCCCC", fontsize=9)

    # Legend
    fig.text(0.30, 0.92, player_a_row["Player"], color="#1A78CF", fontsize=12, fontweight="bold", ha="center")
    fig.text(0.70, 0.92, player_b_row["Player"], color="#EE7300", fontsize=12, fontweight="bold", ha="center")
    plt.show()


In [ ]:
# 7. Interactive widgets: search Player A -> show pizza -> add Player B -> overlay comparison

player_a_input = widgets.Text(
    placeholder="Type Player A name, e.g. 'Bukayo Saka'",
    description="Player A:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "80px"},
)
show_a_button = widgets.Button(description="Show Pizza Chart", button_style="primary", icon="pie-chart")

player_b_input = widgets.Text(
    placeholder="Type Player B name to compare, e.g. 'Mohamed Salah'",
    description="Player B:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "80px"},
)
add_b_button = widgets.Button(description="Add Player B (Compare)", button_style="warning", icon="plus")
add_b_button.disabled = True  # enabled once Player A is found

output_area = widgets.Output()

# state holder
state = {"player_a": None, "player_b": None}


def on_show_a_clicked(b):
    with output_area:
        clear_output(wait=True)
        name = player_a_input.value.strip()
        if not name:
            print("Please type a player name for Player A.")
            return
        row = find_player(name)
        if row is None:
            print(f"No player found matching '{name}'.")
            add_b_button.disabled = True
            return
        state["player_a"] = row
        state["player_b"] = None
        add_b_button.disabled = False
        plot_single_pizza(row)


def on_add_b_clicked(b):
    with output_area:
        clear_output(wait=True)
        if state["player_a"] is None:
            print("Show Player A first.")
            return
        name = player_b_input.value.strip()
        if not name:
            print("Please type a player name for Player B.")
            return
        row = find_player(name)
        if row is None:
            print(f"No player found matching '{name}'.")
            return
        state["player_b"] = row
        plot_comparison_pizza(state["player_a"], row)


show_a_button.on_click(on_show_a_clicked)
add_b_button.on_click(on_add_b_clicked)

ui = widgets.VBox([
    widgets.HBox([player_a_input, show_a_button]),
    widgets.HBox([player_b_input, add_b_button]),
    output_area,
])

display(ui)


---
### Notes
- **Percentiles** are calculated relative to other players in the *same primary position*
  (those with at least `MIN_90S` 90-minute spells played), not the whole league.
- Edit the `STATS` list in the cell above to change which metrics appear on the pizza
  (any numeric column from the CSV works, e.g. `Gls`, `Ast`, `CrdY`, `MP`, etc.).
- Player search is case-insensitive and matches partial names (e.g. typing `"salah"` finds
  `"Mohamed Salah"`).
- To compare a *different* pair, just type a new name in **Player A** and click
  **Show Pizza Chart** again — it resets the comparison.
